In [1]:
import json
import requests
import pandas as pd
import numpy as np
from uuid import uuid4
import os 
from copy import deepcopy
from CustomTranformers.Custom_Transf import preproccesBeforePipe

In [2]:
X_test = pd.read_csv('data/X_test.csv')
y_test = pd.read_csv('data/y_test.csv')

In [3]:
# url="http://127.0.0.1:5000/should_search"
# headers = {'Content-Type': 'application/json'}


# for idx,raw in X_test.iterrows():
#     dataa = raw.to_dict()
#     r = requests.post(url, data=json.dumps(base_request), headers=headers)
#     print(idx)
#     if not  r.ok :
#         break

In [4]:
base_request = X_test[X_test['Legislation'].notna()].iloc[0].to_dict()
base_request 

{'observation_id': '8ba746d9-9ea5-4058-b0b5-6f37e9c2835e',
 'Type': 'person and vehicle search',
 'Date': '2021-09-14T17:40:00+00:00',
 'Part of a policing operation': False,
 'Latitude': 53.721059,
 'Longitude': -1.861924,
 'Gender': 'male',
 'Age range': '25-34',
 'Officer-defined ethnicity': 'white',
 'Legislation': 'misuse of drugs act 1971 (section 23)',
 'Object of search': 'controlled drugs',
 'station': 'west-yorkshire'}

In [5]:
test = pd.read_csv('data/test.csv',)
test.head()

,observation_id,Type,Date,Part of a policing operation,Latitude,Longitude,Gender,Age range,Officer-defined ethnicity,Legislation,Object of search,station
0,1558a55e-3df2-4665-8beb-0f0c5eaa0408,Person search,2022-04-06 18:25:00+00:00,True,NaN,NaN,Male,10-17,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,nottinghamshire
1,eeb891e3-3913-4590-82a9-dc23c212dceb,Person search,2022-04-18 22:24:57+00:00,NaN,NaN,NaN,Male,10-17,Other,Police and Criminal Evidence Act 1984 (section 1),Articles for use in criminal damage,city-of-london
2,898d6606-c55b-4a54-9480-f967beaff1cf,Person and Vehicle search,2022-01-07 20:20:00+00:00,True,NaN,NaN,Male,25-34,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,nottinghamshire
3,3ff08b3c-c1fc-4c9f-97fe-470cf3a61cef,Person search,2022-05-08 20:21:00+00:00,True,NaN,NaN,Male,over 34,White,Misuse of Drugs Act 1971 (section 23),Controlled drugs,nottinghamshire
4,73d7c589-7605-42ab-9c5c-d0fbb897adb0,Person search,2022-02-12 01:43:00+00:00,NaN,54.525683,-1.556382,Male,over 34,White,Police and Criminal Evidence Act 1984 (section 1),Article for use in theft,durham


In [ ]:
# with open(os.path.join('pickles', 'columns.json')) as fh:
#     columns = json.load(fh)
# obs = pd.DataFrame([base_request], columns=columns)
# preproccesBeforePipe(obs.iloc[[0]]).squeeze().to_dict()

In [11]:
APP_NAME = 'model-deploy-production.up.railway.app'
url = "https://{}/should_search/".format(APP_NAME)
# link : https://model-deploy-production.up.railway.app/should_search
r = requests.post(url, json=base_request)
print(r.status_code)
print(r.text)

200
{
  "outcome": true
}



In [12]:
APP_NAME = 'model-deploy-production.up.railway.app'
url = "https://{}/should_search/".format(APP_NAME)
for idx,raw in test.iterrows():
    request = raw.to_dict()
    request = {key: value if pd.notna(value) else None for key, value in request.items()}
    r = requests.post(url, json=request)
    print(isinstance(r, requests.Response))
    print(r.status_code)
    print(r.text)
    if idx >5 : 
        break


True
200
{
  "outcome": false
}

True
200
{
  "outcome": true
}

True
200
{
  "outcome": true
}

True
200
{
  "outcome": false
}

True
200
{
  "outcome": false
}

True
200
{
  "outcome": false
}

True
200
{
  "outcome": true
}



In [ ]:
import sqlite3
conn = sqlite3.connect('prediction.db')
cursor = conn.cursor()
cursor.execute('select * from prediction')
rows = cursor.fetchall()


In [7]:
url="http://127.0.0.1:5000/should_search/"
headers = {'Content-Type': 'application/json'}


r = requests.post(url, data=json.dumps(base_request), headers=headers)
print(r.status_code)
print(r.text)


200
{
  "error": "ERROR: Observation ID: '8ba746d9-9ea5-4058-b0b5-6f37e9c2835e' already exists",
  "outcome": true
}



In [13]:
bad_request_1 = deepcopy(base_request)
bad_request_1['random_field'] = bad_request_1.pop('observation_id')


bad_request_2 = deepcopy(base_request)
bad_request_2['data_field_name'] = bad_request_2.pop('Type')



bad_request_3 = deepcopy(base_request)
bad_request_3.pop('Age range')


bad_request_4 = deepcopy(base_request)
bad_request_4['relationship'] = "Wife"

bad_request_5 = deepcopy(base_request)
bad_request_5['Gender'] = "Engineer"


bad_request_6 = deepcopy(base_request)
bad_request_6['Latitude'] = 'Male'


bad_request_7 = deepcopy(base_request)
bad_request_7['Latitude'] = -12


bad_request_8 = deepcopy(base_request)
bad_request_8['Age range'] = 1200


bad_request_9 = deepcopy(base_request)
bad_request_9['Date'] = '10-May'


bad_request_10 = deepcopy(base_request)
bad_request_10['observation_id'] =f'{uuid4()}'
bad_request_10['Date'] = '5-may-2020'


bad_request_11 = deepcopy(base_request)
bad_request_11['Officer-defined ethnicity'] = -10


bad_request_12 = deepcopy(base_request)
bad_request_12['observation_id'] = f'{uuid4()}'
bad_request_12['Object of search'] = "rubbery"

# for x in [bad_request_1,bad_request_2,bad_request_3,bad_request_4,bad_request_5,bad_request_6,bad_request_7,bad_request_8,bad_request_9,bad_request_10
#          ,bad_request_11,bad_request_12]:
#     r = requests.post(url, data=json.dumps(x), headers=headers)

#     print(r.status_code)
#     print(r.text)
# railway
for x in [bad_request_1,bad_request_2,bad_request_3,bad_request_4,bad_request_5,bad_request_6,bad_request_7,bad_request_8,bad_request_9,bad_request_10
         ,bad_request_11,bad_request_12]:
    r = requests.post(url, json=x)

    print(r.status_code)
    print(r.text)

200
{
  "error": "Field `observation_id` missing from request: {'Type': 'person and vehicle search', 'Date': '2021-09-14T17:40:00+00:00', 'Part of a policing operation': False, 'Latitude': 53.721059, 'Longitude': -1.861924, 'Gender': 'male', 'Age range': '25-34', 'Officer-defined ethnicity': 'white', 'Legislation': 'misuse of drugs act 1971 (section 23)', 'Object of search': 'controlled drugs', 'station': 'west-yorkshire', 'random_field': '8ba746d9-9ea5-4058-b0b5-6f37e9c2835e'}",
  "observation_id": null
}

200
{
  "error": "Missing columns: {'Type'}",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "Missing columns: {'Age range'}",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "Unrecognized columns provided: {'relationship'}",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "Invalid value provided for Gender: engineer. Allowed values are: 'female','male','other'",
  "observation_id": "8ba746

In [14]:


bad_request_13 = deepcopy(base_request)
bad_request_13['observation_id'] = f'{uuid4()}'
bad_request_13['Type'] = "person"

bad_request_14 = deepcopy(base_request)
bad_request_14['observation_id'] = f'{uuid4()}'
bad_request_14['station'] = "hampshirre"

bad_request_15 = deepcopy(base_request)
bad_request_15['observation_id'] = f'{uuid4()}'
bad_request_15['Latitude'] = int(55)

bad_request_16 = deepcopy(base_request)
bad_request_16['observation_id'] = f'{uuid4()}'
bad_request_16['Part of a policing operation'] = None

bad_request_17 = deepcopy(base_request)
bad_request_17['observation_id'] = f'{uuid4()}'
bad_request_17['Part of a policing operation'] = 'any'

bad_request_18 = deepcopy(base_request)
bad_request_18['observation_id'] = f'{uuid4()}'
bad_request_18['Legislation'] = None

# for x in [bad_request_13,bad_request_14,bad_request_15,bad_request_16,bad_request_17]:
#     r = requests.post(url, data=json.dumps(x), headers=headers)

#     print(r.status_code)
#     print(r.text)
for x in [bad_request_13,bad_request_14,bad_request_15,bad_request_16,bad_request_17,bad_request_18]:
    r = requests.post(url, json=x)

    print(r.status_code)
    print(r.text)

200
{
  "error": "the value provided for Type: person. is close to `person search ` may be u misspelling it ",
  "observation_id": "06cb36d4-40f0-4d36-893a-1954512a1654"
}

200
{
  "error": "the value provided for station: hampshirre. is close to `hampshire ` may be u misspelling it ",
  "observation_id": "94e50090-6a87-43ab-8f5a-ed1a9cf266f1"
}

200
{
  "error": "Field `Latitude` must be float value not integer",
  "observation_id": "b1fe1026-2899-4340-bd7b-9d7552859b54"
}

200
{
  "outcome": true
}

200
{
  "error": "Field `Part of a plocinig operation` should be boolen get only true or false ",
  "observation_id": "e844802d-fc59-4295-a4de-67c938f058a3"
}

200
{
  "outcome": true
}



# the Database view

In [ ]:
# APP_NAME = 'model-deploy-production-15ec.up.railway.app'
# url = "https://{}/list_db_contents".format(APP_NAME)
# r = requests.post(url)
# print(isinstance(r, requests.Response))
# print(r.status_code)
# print(r.text)

In [ ]:
url="http://127.0.0.1:5000/list-db-contents"
headers = {'Content-Type': 'application/json'}

r = requests.post(url, headers=headers)
print(r.status_code)
print(r.text)

# testcases for search_result

In [15]:
APP_NAME = 'model-deploy-production.up.railway.app'
url = "https://{}/search_result/".format(APP_NAME)
update_request = {'observation_id':"8ba746d9-9ea5-4058-b0b5-6f37e9c2835e",'outcome': True}
r = requests.post(url, json=update_request)
print(isinstance(r, requests.Response))
print(r.status_code)
print(r.text)

True
200
{
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e",
  "outcome": true,
  "predicted_outcome": true
}



In [ ]:
# url="http://127.0.0.1:5000/search_result"
# headers = {'Content-Type': 'application/json'}
# update_request = {'observation_id':"8ba746d9-9ea5-4058-b0b5-6f37e9c2835e",'outcome': True}
# r = requests.post(url, data=json.dumps(update_request), headers=headers)
# print(r.status_code)
# print(r.text)

In [16]:
bad_request_1 = deepcopy(update_request)
bad_request_1['outcome'] = "true"

bad_request_2 = deepcopy(update_request)
bad_request_2['outcome'] = "any"

bad_request_3 = deepcopy(update_request)
bad_request_3['outcome'] = None

bad_request_4 = deepcopy(update_request)
bad_request_4.pop('outcome')

bad_request_5 = deepcopy(update_request)
bad_request_5.pop('outcome')

bad_request_5 = deepcopy(update_request)
bad_request_5.pop('observation_id')

# for x in [bad_request_1,bad_request_2,bad_request_3,bad_request_4,bad_request_5]:
#     r = requests.post(url, data=json.dumps(x), headers=headers)

#     print(r.status_code)
#     print(r.text)
for x in [bad_request_1,bad_request_2,bad_request_3,bad_request_4,bad_request_5]:
    r = requests.post(url,json= x )

    print(r.status_code)
    print(r.text)

200
{
  "error": "Field `outcome` u have enter good value but should be in boolen not string",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "Field `outcome` should be boolen get only true or false ",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "U have enter None value to outcome and it should be boolen [true , false]",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "Missing columns: {'outcome'}",
  "observation_id": "8ba746d9-9ea5-4058-b0b5-6f37e9c2835e"
}

200
{
  "error": "Field `observation_id` missing from request: {'outcome': True}",
  "observation_id": null
}

